# Stage 04: Synthetic in-domain transcripts  `[GPU-light]`
Paper §4.1 Step 1 — few-shot an open LLM for new in-domain transcripts, with the
n-gram leakage guard rejecting near-copies. `synth_count=None` (full) matches the
real train size (nsyn = n).

In [ ]:
# --- CarePath stage bootstrap (short by design) ---
import importlib.util, os, subprocess, sys
from pathlib import Path

def _find(start):
    for d in [start, *start.parents]:
        if (d / 'pyproject.toml').exists() and (d / 'apps' / 'api' / 'carepath').exists():
            return d
    return None

REPO = _find(Path.cwd().resolve())
if REPO is None and importlib.util.find_spec('google.colab'):
    url = os.environ.get('CAREPATH_REPO_URL', 'https://github.com/truong-tt/carepath.git')
    tok = os.environ.get('CAREPATH_GITHUB_TOKEN') or os.environ.get('GITHUB_TOKEN')
    if tok and url.startswith('https://github.com/'):
        url = url.replace('https://', f'https://x-access-token:{tok}@')
    subprocess.run(['git', 'clone', url, '/content/carepath'], check=True)
    REPO = Path('/content/carepath')
assert REPO, 'Open this notebook from inside the CarePath repo.'
os.chdir(REPO); sys.path.insert(0, str(REPO / 'apps' / 'api'))

PROFILE = 'smoke'   # <<< set to 'full' for the real ViMedCSS run
from carepath.gec.notebook import init_stage
CTX = init_stage(PROFILE); P = CTX.paths; PROF = CTX.profile


In [ ]:
# Install the GEC training stack (idempotent; needed once per Colab runtime).
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[training]'])


In [ ]:
CTX.restore([str(P.real_pairs)])
from carepath.gec.data import read_jsonl
count = PROF.synth_count or (sum(1 for r in read_jsonl(P.real_pairs) if r.get('split') == 'train') or 50)
args = ['scripts/gec/gen_synthetic.py', '--pairs', str(P.real_pairs),
        '--output', str(P.synth_clean), '--count', str(count)]
if PROF.name != 'smoke':
    args.append('--load-in-4bit')
CTX.run_step(args)
